# Sentinel-1 Analysis Notebook

This notebook explores Sentinel-1 SAR data in the Ghana parquet dataset, including:

1. S1 temporal trends (2017-2023, quarterly)
2. S1 relationships with S2 indices (NDVI/NDMI/NDBI/NDWI/BSI)
3. Paved vs unpaved comparisons for S1 metrics

It is robust to missing columns and will print guidance if required fields are unavailable.


In [2]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')


In [3]:
from pathlib import Path
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pandas as pd

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
PARQ_DIR = DATA_DIR / 'ghana_parquet_s1'

needed = [
    'osm_id', 'fclass', 'year', 'quarter',
    'NDVI', 'NDMI', 'NDBI', 'NDWI', 'BSI',
    's1_vv_mean', 's1_vh_mean', 's1_vv_minus_vh_mean',
    's1_vv_std', 's1_vh_std', 's1_vv_minus_vh_std'
]

dataset = ds.dataset(str(PARQ_DIR), format='parquet', partitioning='hive')
avail = set(dataset.schema.names)
cols = [c for c in needed if c in avail]

table = dataset.to_table(columns=cols)

# Force-decode any dictionary columns to their value dtype
for i, field in enumerate(table.schema):
    col = table.column(i)
    if pa.types.is_dictionary(field.type):
        target_type = field.type.value_type  # e.g. int32, string
        chunks = [pc.cast(chunk, target_type) for chunk in col.chunks]
        new_col = pa.chunked_array(chunks, type=target_type)
        table = table.set_column(i, field.name, new_col)

df = table.to_pandas()

# standardize keys
df['osm_id'] = df['osm_id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df['quarter'] = df['quarter'].astype(str).str.upper().str.extract(r'(Q[1-4])', expand=False)

print("Rows:", len(df), "| Cols:", len(df.columns))
print(df.columns.tolist())

Rows: 37517856 | Cols: 15
['osm_id', 'fclass', 'year', 'quarter', 'NDVI', 'NDMI', 'NDBI', 'NDWI', 'BSI', 's1_vv_mean', 's1_vh_mean', 's1_vv_minus_vh_mean', 's1_vv_std', 's1_vh_std', 's1_vv_minus_vh_std']


In [4]:
# Standardize key columns + load paved/unpaved from latest OSM PBF in data/
import numpy as np
from pyrosm import OSM

pbf_candidates = sorted(DATA_DIR.glob('*.osm.pbf'))
PBF_PATH = pbf_candidates[-1]
print('PBF used:', PBF_PATH.name)

df['osm_id'] = df['osm_id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df['quarter'] = df['quarter'].astype(str).str.upper().str.extract(r'(Q[1-4])', expand=False)

osm = OSM(str(PBF_PATH))
roads = osm.get_network(network_type='driving').copy()

roads['osm_id'] = roads['id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
roads['surface_clean'] = roads['surface'].astype(str).str.lower().str.strip()

paved_vals = {
    'paved', 'asphalt', 'concrete', 'paving_stones', 'concrete:plates',
    'sett', 'cobblestone', 'metal', 'bricks', 'cement', 'chipseal'
}
unpaved_vals = {
    'unpaved', 'ground', 'dirt', 'earth', 'gravel', 'fine_gravel',
    'sand', 'mud', 'grass', 'compacted', 'pebblestone', 'soil'
}

roads['surface_class'] = pd.Series(pd.NA, index=roads.index, dtype='string')
roads.loc[roads['surface_clean'].isin(paved_vals), 'surface_class'] = 'paved'
roads.loc[roads['surface_clean'].isin(unpaved_vals), 'surface_class'] = 'unpaved'

surface_df = roads[['osm_id', 'surface_class']].dropna(subset=['surface_class']).drop_duplicates(subset=['osm_id'])

work = df.merge(surface_df, on='osm_id', how='left')
work['paved_label'] = work['surface_class']

print('Surface rows from PBF:', len(surface_df))
print('Merged rows:', len(work))
print('Surface coverage:', round(float(work['paved_label'].notna().mean()), 4))
print(work['paved_label'].value_counts(dropna=False).head(20))



PBF used: ghana-260415.osm.pbf
Surface rows from PBF: 30464
Merged rows: 37517856
Surface coverage: 0.0901
paved_label
<NA>       34138554
unpaved     1928082
paved       1451220
Name: count, dtype: Int64


In [5]:
# Detect S1 and S2 columns
s1_cols = [c for c in df.columns if c.startswith('s1_')]
s2_idx_cols = [c for c in ['NDVI','NDMI','NDBI','NDWI','BSI'] if c in df.columns]

# Also detect quarterly S2 columns (e.g., ndvi_2020_Q1)
qpat = re.compile(r'^(ndvi|ndmi|ndbi|ndwi|bsi)_(\d{4})_Q([1-4])$', re.I)
s2_quarterly = [c for c in df.columns if qpat.match(str(c))]

print('S1 columns:', len(s1_cols))
print(s1_cols[:20])
print('S2 base cols:', s2_idx_cols)
print('S2 quarterly cols:', len(s2_quarterly))


S1 columns: 6
['s1_vv_mean', 's1_vh_mean', 's1_vv_minus_vh_mean', 's1_vv_std', 's1_vh_std', 's1_vv_minus_vh_std']
S2 base cols: ['NDVI', 'NDMI', 'NDBI', 'NDWI', 'BSI']
S2 quarterly cols: 0


In [ ]:
# Keep analysis frame with available essentials
need = [c for c in ['osm_id','year','quarter','fclass','road_class','paved_label'] if c in df.columns]
metric_cols = s1_cols + s2_idx_cols
work = df[need + metric_cols].copy()

# numeric cast for metrics
for c in metric_cols:
    work[c] = pd.to_numeric(work[c], errors='coerce')

print('Analysis rows:', len(work))
print('NA share (first metrics):')
display(work[metric_cols].isna().mean().sort_values().head(15) if metric_cols else pd.Series(dtype=float))


## 1) Sentinel-1 Temporal Trends


In [ ]:
# Choose common S1 metrics if available
pref_s1 = ['s1_vv_mean','s1_vh_mean','s1_vv_minus_vh_mean','s1_vv_std','s1_vh_std','s1_vv_minus_vh_std']
avail_s1 = [c for c in pref_s1 if c in work.columns]

# Ensure time keys exist
if 'year' not in work.columns and 'year' in df.columns:
    work['year'] = pd.to_numeric(df['year'], errors='coerce')
if 'quarter' not in work.columns and 'quarter' in df.columns:
    work['quarter'] = df['quarter'].astype(str).str.upper().str.extract(r'(Q[1-4])', expand=False)

if 'year' not in work.columns or 'quarter' not in work.columns:
    raise ValueError(f"Missing time columns. Found columns: {work.columns.tolist()[:40]}")

if not avail_s1:
    print('No expected S1 metric columns found. Available s1 columns:')
    print([c for c in work.columns if c.startswith('s1_')][:50])
else:
    trend = (
        work
        .dropna(subset=['year','quarter'])
        .groupby(['year','quarter'])[avail_s1]
        .mean(numeric_only=True)
        .reset_index()
    )

    trend['year'] = pd.to_numeric(trend['year'], errors='coerce')
    trend['qnum'] = trend['quarter'].str.extract(r'Q([1-4])', expand=False).astype(float)
    trend = trend.sort_values(['year','qnum'])
    trend['t'] = np.arange(len(trend))

    display(trend.head(12))

    for col in avail_s1:
        plt.figure(figsize=(12,4))
        sns.lineplot(data=trend, x='t', y=col, marker='o')
        plt.title(f'S1 Trend: {col} (quarterly mean)')
        plt.xlabel('Time index (year-quarter sorted)')
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()

KeyError: ['year']

## 2) Relationship Between S1 and S2


In [ ]:
# Build S2 from quarterly if base cols are missing
rel = work.copy()

if not s2_idx_cols and s2_quarterly:
    for idx in ['ndvi','ndmi','ndbi','ndwi','bsi']:
        cols = [c for c in s2_quarterly if c.lower().startswith(idx+'_')]
        if cols:
            rel[idx.upper()] = rel[cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)
    s2_idx_cols = [c for c in ['NDVI','NDMI','NDBI','NDWI','BSI'] if c in rel.columns]

avail_s1_core = [c for c in ['s1_vv_mean','s1_vh_mean','s1_vv_minus_vh_mean'] if c in rel.columns]

if not avail_s1_core or not s2_idx_cols:
    print('Insufficient columns for S1↔S2 relationship analysis.')
    print('S1 core:', avail_s1_core)
    print('S2 cols:', s2_idx_cols)
else:
    pair_cols = avail_s1_core + s2_idx_cols
    corr = rel[pair_cols].corr(method='spearman')
    plt.figure(figsize=(10,8))
    sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f')
    plt.title('Spearman Correlation: S1 vs S2')
    plt.tight_layout()
    plt.show()

    # selected scatter plots
    combos = []
    for s1 in avail_s1_core:
        for s2 in s2_idx_cols:
            combos.append((s1,s2))

    for s1,s2 in combos[:9]:
        d = rel[[s1,s2]].dropna()
        if len(d) < 200:
            continue
        d = d.sample(min(len(d), 4000), random_state=42)
        plt.figure(figsize=(5,4))
        sns.scatterplot(data=d, x=s1, y=s2, s=12, alpha=0.35)
        sns.regplot(data=d, x=s1, y=s2, scatter=False, color='black', line_kws={'lw':1.5})
        plt.title(f'{s2} vs {s1}')
        plt.tight_layout()
        plt.show()


## 3) Paved vs Unpaved Comparisons (S1)


In [ ]:
s1_compare = [c for c in ['s1_vv_mean','s1_vh_mean','s1_vv_minus_vh_mean','s1_vv_std','s1_vh_std'] if c in work.columns]
sub = work[work['paved_label'].isin(['paved','unpaved'])].copy()

if sub.empty:
    print('No paved/unpaved labels available for comparison.')
elif not s1_compare:
    print('No S1 comparison columns found.')
else:
    print('Rows for paved/unpaved comparison:', len(sub))

    # summary table
    summ = (
        sub
        .groupby('paved_label')[s1_compare]
        .agg(['mean','median','std','count'])
    )
    display(summ)

    # box plots
    for col in s1_compare:
        plt.figure(figsize=(6,4))
        sns.boxplot(data=sub, x='paved_label', y=col)
        plt.title(f'{col}: paved vs unpaved')
        plt.tight_layout()
        plt.show()


## 4) Optional: Save analysis-ready subset


In [ ]:
OUT = DATA_DIR / 'sentinel1_analysis_subset.csv'
keep = [c for c in ['osm_id','year','quarter','fclass','road_class','paved_label'] if c in work.columns] + [c for c in work.columns if c.startswith('s1_')] + [c for c in ['NDVI','NDMI','NDBI','NDWI','BSI'] if c in work.columns]
work[keep].to_csv(OUT, index=False)
print('Saved:', OUT)
